In [1]:
!pip install -q accelerate peft bitsandbytes transformers trl

In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
    EarlyStoppingCallback
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import pandas as pd
import numpy as np

In [3]:
from google.colab import userdata
sec_key = userdata.get('HF_TOKEN')

In [4]:
!huggingface-cli login --token $sec_key

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [5]:
# The model that you want to train from the Hugging Face hub
model_name = "nvidia/Nemotron-Mini-4B-Instruct"

In [6]:
# Reload model in FP16 and merge it with LoRA weights
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [7]:
pipeline = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

In [9]:
messages = [
    {"role": "user", "content": """
    A partir de ahora eres un profesor que habla español.
    En cuál de las reacciones siguientes participa la biotina.
     a) Hidroxilaciones.
     b)Carboxilaciones.
     c) Deshidrataciones.
     d) Desaminaciones.
    """},
]

outputs = pipeline(
    messages,
    max_new_tokens=300,
)
print(outputs[0]["generated_text"][-1].get('content'))

 La respuesta correcta es a) Hidroxilaciones. La biotina, o ácido tiocianato, participa en reacciones hidroxiladas, específicamente en la hidroxilación de aldehídos y cetonas. La biotina actúa como un catalizador en estas reacciones, acelerando la formación de alcoholes y aldehídos hidroxilados. Por lo tanto, la biotina no participa en las reacciones b) carboxilaciones, c) deshidrataciones, ni d) desaminaciones.


In [13]:
messages = [
    {"role": "user", "content": """A partir de ahora eres un profesor que habla español.
    Quiero que me hagas entre tres y cinco preguntas acerca del tema a
    examinar, nivel de complejidad o tipo de preguntas, es decir, de desarrollo,
    verdadero o falso u opción múltiple. Añade alguna otra pregunta si que consideres necesaria.
    Cuando tengas suficiente información genera un archivo de texto con las preguntas.
    """},
]

outputs = pipeline(
    messages,
    max_new_tokens=300,
)


In [14]:
messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})

In [15]:
messages

[{'role': 'user',
  'content': 'A partir de ahora eres un profesor que habla español.\n    Quiero que me hagas entre tres y cinco preguntas acerca del tema a\n    examinar, nivel de complejidad o tipo de preguntas, es decir, de desarrollo,\n    verdadero o falso u opción múltiple. Añade alguna otra pregunta si que consideres necesaria.\n    Cuando tengas suficiente información genera un archivo de texto con las preguntas.\n    '},
 {'role': 'assistant',
  'content': ' ¡Claro, estaré encantado de ayudarte con eso! ¿Podrías indicar el tema en particular que deseas evaluar? Esto me ayudará a crear preguntas más específicas y relevantes.'}]

In [16]:
print(outputs[0]["generated_text"][-1].get('content'))

 ¡Claro, estaré encantado de ayudarte con eso! ¿Podrías indicar el tema en particular que deseas evaluar? Esto me ayudará a crear preguntas más específicas y relevantes.


In [17]:
messages.append({"role": "user", "content": """1. Tema: álgebra lineal"""})

outputs = pipeline(
    messages,
    max_new_tokens=300,
)
messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})

In [18]:
print(outputs[0]["generated_text"][-1].get('content'))

 ¡Excelente! Aquí te proporciono algunas preguntas sobre álgebra lineal, con diferentes niveles de complejidad y tipos de preguntas. Puedes elegir entre ellas o solicitar que se añadan más preguntas según tus necesidades.

1. **Pregunta de desarrollo:** Explica la diferencia entre vectores y matrices en términos de dimensión y operaciones.
2. **Pregunta de verdad o falso:** La matriz identidad es una matriz cuadrada con todos sus elementos en 1 y 0 en las demás posiciones. (Verdadero)
3. **Pregunta de verdad o falso:** La matriz identidad es una matriz inversa de cualquier matriz cuadrada. (Falso)
4. **Pregunta de verdad o falso:** La matriz identidad es una matriz adyacente a cualquier matriz cuadrada. (Verdadero)
5. **Pregunta de verdad o falso:** La matriz identidad es una matriz diagonal dominante. (Falso)
6. **Pregunta de verdad o falso:** La matriz identidad es una matriz triangular superior. (Verdadero)
7. **Pregunta de verdad o falso:** La matriz identidad es una matriz triangu

In [19]:
messages.append({"role": "user", "content": """nivel de complejidad: intermedio"""})

outputs = pipeline(
    messages,
    max_new_tokens=300,
)
messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})

In [20]:
print(outputs[0]["generated_text"][-1].get('content'))

 ¡Excelente elección! Aquí te proporciono algunas preguntas sobre álgebra lineal de un nivel intermedio, con diferentes tipos de preguntas. Puedes elegir entre ellas o solicitar que se añadan más preguntas según tus necesidades.

1. **Pregunta de desarrollo:** Explica cómo se calcula el producto punto entre dos vectores y cómo se utiliza en la geometría lineal.
2. **Pregunta de verdad o falso:** La matriz identidad es una matriz inversa de cualquier matriz cuadrada si y solo si la matriz es no singular. (Verdadero)
3. **Pregunta de verdad o falso:** La matriz identidad es una matriz adyacente a cualquier matriz cuadrada si y solo si la matriz es no singular y tiene todos sus elementos en 0 o 1. (Falso)
4. **Pregunta de verdad o falso:** La matriz identidad es una matriz diagonal dominante si y solo si todos sus elementos en la diagonal principal son 1 y los demás son 0. (Verdadero)
5. **Pregunta de verdad o falso:** La matriz identidad es una matriz triangular superior si y solo si tod

In [21]:
messages.append({"role": "user", "content": """3 preguntas de opción múltiple, 2 de verdadero o falso y 2 de desarrollo"""})

outputs = pipeline(
    messages,
    max_new_tokens=1500,
)
messages.append({"role": "assistant", "content": outputs[0]["generated_text"][-1].get('content')})

In [22]:
print(outputs[0]["generated_text"][-1].get('content'))

 ¡Claro! Aquí te proporciono tres preguntas de opción múltiple, dos preguntas de verdad o falso y dos preguntas de desarrollo sobre álgebra lineal. Puedes elegir entre ellas o solicitar que se añadan más preguntas según tus necesidades.

**Preguntas de opción múltiple:**

1. ¿Cuál de los siguientes vectores es ortogonal al vector $(1, 2, 3)$?
   a) $(1, 1, 1)$
   b) $(1, 1, 1)$
   c) $(-1, -1, -1)$
   d) $(1, -1, 1)$

2. ¿Cuál de los siguientes vectores es ortogonal al vector $(1, 2, 3)$?
   a) $(1, 1, 1)$
   b) $(1, -1, 1)$
   c) $(-1, -1, -1)$
   d) $(1, 1, -1)$

**Preguntas de verdad o falso:**

1. (F) La matriz identidad es una matriz inversa de cualquier matriz cuadrada si y solo si la matriz es no singular.
2. (T) La matriz identidad es una matriz adyacente a cualquier matriz cuadrada si y solo si la matriz es no singular y tiene todos sus elementos en 0 o 1.

**Preguntas de desarrollo:**

1. Explica cómo se calcula el producto punto entre dos vectores y cómo se utiliza en la geo